In [1]:
import numpy as np
import pandas as pd
import time
import os
import random
import joblib

import elkai
import tsplib95
from scipy.spatial.distance import cdist
from scipy.stats import skew, kurtosis
from scipy.sparse.csgraph import minimum_spanning_tree

from mealpy import GA, SA, PSO, ACOR
from mealpy.utils.problem import Problem
from mealpy.utils.space import FloatVar

class TSPProblem(Problem):
    def __init__(self, D, **kwargs):
        self.D = D
        self.n = len(D)
        bounds = FloatVar(lb=[0.0] * self.n, ub=[self.n - 1.0] * self.n)
        super().__init__(
            bounds=bounds,
            minmax="min",
            **kwargs
        )

    def obj_func(self,x):
        tour = np.argsort(x).tolist()
        cost = sum(self.D[tour[i]][tour[i+1]] for i in range(self.n - 1))
        cost += self.D[tour[-1]][tour[0]]
        return cost
    


In [2]:
def run_mealpy(model,D):
    problem = TSPProblem(D=D, log_to=None)
    t0 = time.time()
    model.solve(problem)
    runtime = time.time() - t0
    cost = model.g_best.target.fitness
    history = model.history.list_global_best_fit
    return cost, runtime, history

def run_lk(D):
    D_int = (D * 1000).astype(int).tolist()
    t0 = time.time()
    tour = elkai.solve_int_matrix(D_int)
    runtime = time.time() - t0
    cost = sum(D[tour[i]][tour[i+1]] for i in range(len(tour) - 1)) + D[tour[-1]][tour[0]]
    return cost, runtime, None

In [3]:
ALGORITHM_POOL = {
    "GA":  lambda D: run_mealpy(GA.BaseGA(epoch=500, pop_size=100), D),
    "SA":  lambda D: run_mealpy(SA.OriginalSA(epoch=500, pop_size=10), D),
    "LK":  lambda D: run_lk(D),
    "PSO": lambda D: run_mealpy(PSO.OriginalPSO(epoch=500, pop_size=100), D),
    "ACO": lambda D: run_mealpy(ACOR.OriginalACOR(epoch=500, pop_size=100), D),
}


In [4]:
def compute_distance_matrix(coords, distance_type="euclidean"):
    n = len(coords)
    D = np.zeros((n,n))

    if distance_type == "euclidean":
        D = cdist(coords, coords, metrix='euclidean')

    elif distance_type == "manhattan":
        D = cdist(coords, coords, metric="cityblock")

    elif distance_type == "chebyshev":
        D = cdist(coords, coords, metric="chebyshev")

    elif distance_type == "ceil_euclidean":
        D = np.ceil(cdist(coords, coords, metric="euclidean"))

    else:
        raise ValueError(f"Unsupported distance type: {distance_type}")
        
    np.fill_diagonal(D, 0)
    return D

DISTANCE_TYPE_ENCODING = {
    "euclidean": 0,
    "manhattan": 1,
    "chebyshev": 2,
    "ceil_euclidean": 3
}

In [5]:
def generate_instance(n, distribution, seed, distance_type="euclidean"):
    rng = np.random.default_rng(seed)

    if distribution == "uniform":
        coords = rng.uniform(0, 1000, (n,2))

    elif distribution == "clustered":
        centers = rng.uniform(100, 900, (max(2, n//10),2))
        coords = np.array([centers[i % max(2, n//10)] + rng.normal(0,30,2) for i in range(n)])

    elif distribution == "grid":
        side = int(np.ceil(np.sqrt(n)))
        coords = np.array([[x,y] for x in range(side) for y in range(side)])[:n] * (1000/side)

    elif distribution == "diagonal":
        t = rng.uniform(0, 1000, n)
        noise = rng.normal(0,20,(n,2))
        coords = np.column_stack([t, t]) + noise

    D = cdist(coords, coords, distance_type)
    return coords, D

In [6]:
def load_tsplib(name, folder="./tsplib"):
    problem = tsplib95.load(os.path.join(folder, f"{name}.tsp"))
    n = problem.dimension
    nodes = list(problem.get_nodes())

    D = np.zeros((n, n))
    for i in nodes:
        for j in nodes:
            D[i-1][j-1] = problem.get_weight(i, j)

    coords = None
    if problem.node_coords:
        coords = np.array([problem.node_coords[i] for i in nodes])

    return coords, D

In [7]:
def extract_features(coords, D):
    n = len(D)
    upper = D[np.triu_indices(n, k=1)]

    D_nn = D.copy()
    np.fill_diagonal(D_nn, np.inf)
    nn = D_nn.min(axis=1)

    mst_arr = minimum_spanning_tree(D).toarray()
    mst_vals = mst_arr[mst_arr > 0]

    features = {
        "n_cities": n,
        "avg_edge": upper.mean(),
        "std_edge": upper.std(),
        "cv_edge": upper.std() / upper.mean(),
        "skew_edge": float(skew(upper)),
        "kurtosis_edge": float(kurtosis(upper)),
        "min_edge": upper.min(),
        "max_edge": upper.max(),

        "avg_nn": nn.mean(),
        "std_nn": nn.std(),
        "cv_nn": nn.std() / nn.mean(),
        "skew_nn": float(skew(nn)),

        "mst_cost": mst_vals.sum(),
        "avg_mst_edge": mst_vals.mean(),
        "std_mst_edge": mst_vals.std(),
        "mst_nn_ratio": mst_vals.mean() / nn.mean()
    }

    if coords is not None:
        centroid = coords.mean(axis=0)
        centroid_dists = np.linalg.norm(coords - centroid, axis=1)

        x_range = coords[:, 0].max() - coords[:, 0].min()
        y_range = coords[:, 1].max() - coords[:, 1].min()

        features.update({
            "avg_centroid_dist": centroid_dists.mean(),
            "std_centroid_dist": centroid_dists.std(),
            "cv_centroid_dist": centroid_dists.std() / centroid_dists.mean(),
            "bbox_area": x_range * y_range,
            "bbox_ratio": x_range / (y_range +1e-9)
        })

    else:
        for col in ["avg_centroid_dist","std_centroid_dist",
                    "cv_centroid_dist","bbox_area","bbox_ratio"]:
            features[col] = np.nan
        
    return features


In [8]:
def solve_concorde(coords):
    try:
        from concorde.tsp import TSPSolver
        solver = TSPSolver.from_data(
            coords[:, 0].tolist(),
            coords[:, 1].tolist(),
            norm = "EUC_2D"
        )
        solution = solve.solver(verbose=False)
        tour = list(solution.tour)
        D = cdist(coords, coords)
        cost = sum(D[tour[i]][tour[i_1]] for i in range(len(tour)-1)) + D[tour[-1]][tour[0]]
        
        return cost
    except ImportError:
        return None

In [9]:
def run_all(D, coords=None, n_runs=3, use_concorde=False):
    results = {}

    for name, algo_fn  in ALGORITHM_POOL.items():
        run_costs = []
        run_times = []

        for _ in range(n_runs):
            cost, runtime, _ = algo_fn(D)
            run_costs.append(cost)
            run_times.append(runtime)
        results[name] = {
            "best_cost": min(run_costs),
            "avg_cost": np.mean(run_costs),
            "std_cost": np.std(run_costs),
            "avg_time": np.mean(run_times),
        }

    optimal = None
    if use_concorde and coords is not None and len(D) <= 150:
        optimal = solve_concorde(coords)

    return results, optimal

In [10]:
def build_row(coords, D, source, label, n_runs=3, use_concorde=False):
    features = extract_features(coords, D)
    algo_results, optimal = run_all(
        D, coords, n_runs=n_runs, use_concorde=use_concorde
    )

    best_algo = min(algo_results, key=lambda k: algo_results[k]["best_cost"])

    row = {
        **features,
        "source": source,
        "label": label,
        "best_algo": best_algo,
        "optimal": optimal,
    }

    for algo, metrics in algo_results.items():
        for metric, val in metrics.items():
            row[f"{algo}_{metric}"] = val

        if optimal:
            row[f"{algo}_gap"] = (
                (algo_results[algo]["best_cost"] - optimal / optimal)
            )
        return row
    
def build_dataset(configs, source="generated", use_concorde=False):
    rows=[]
    for i, cfg in enumerate(configs):
        if source == "generated":
            coords, D = generate_instance(
                cfg["n"], cfg["distribution"], cfg["seed"], cfg.get("distance_type","euclidean")
            )
            label = f"{cfg['distribution']}_{cfg['n']}_{cfg['seed']}"
        else:
            coords, D = load_tsplib(cfg["name"])
            label = cfg["name"]
        
        row = build_row(
            coords, D,
            source=source,
            label=label,
            use_concorde=use_concorde
        )
        rows.append(row)
        print(f"[{i+1}/{len(configs)}] {label} -> best: {row['best_algo']}")
    
    return pd.DataFrame(rows)



In [11]:
train_configs = [
    {"n": n, "distribution": d, "seed": s}
    for n in [20,30,50,75,100,150,200]
    for d in ["uniform","clustered", "grid","diagonal"]
    for s in range(15)
    for dt in ["euclidean","manhattan","ceil_euclidean","chebyshev"]
]

TSPLIB_PROBLEMS = [
    "eil51", "berlin52", "st70", "eil76",
    "pr76", "kroA100", "eil101", "lin105"
]
val_configs = [{"name": p} for p in TSPLIB_PROBLEMS]
train_df = build_dataset(train_configs, source="generated", use_concorde=False)
val_df = build_dataset(val_configs, source="tsplib", use_concorde=True)

train_df.to_csv("train_meta.csv", index=False)
val_df.to_csv("val_meta.csv", index=False)

[1/1680] uniform_20_0 -> best: LK
[2/1680] uniform_20_0 -> best: LK
[3/1680] uniform_20_0 -> best: LK
[4/1680] uniform_20_0 -> best: LK
[5/1680] uniform_20_1 -> best: LK
[6/1680] uniform_20_1 -> best: LK
[7/1680] uniform_20_1 -> best: GA
[8/1680] uniform_20_1 -> best: LK
[9/1680] uniform_20_2 -> best: LK
[10/1680] uniform_20_2 -> best: LK
[11/1680] uniform_20_2 -> best: GA
[12/1680] uniform_20_2 -> best: LK
[13/1680] uniform_20_3 -> best: GA
[14/1680] uniform_20_3 -> best: LK
[15/1680] uniform_20_3 -> best: LK
[16/1680] uniform_20_3 -> best: GA
[17/1680] uniform_20_4 -> best: GA
[18/1680] uniform_20_4 -> best: LK
[19/1680] uniform_20_4 -> best: GA
[20/1680] uniform_20_4 -> best: GA
[21/1680] uniform_20_5 -> best: LK
[22/1680] uniform_20_5 -> best: GA
[23/1680] uniform_20_5 -> best: GA
[24/1680] uniform_20_5 -> best: LK
[25/1680] uniform_20_6 -> best: LK
[26/1680] uniform_20_6 -> best: LK
[27/1680] uniform_20_6 -> best: GA
[28/1680] uniform_20_6 -> best: GA
[29/1680] uniform_20_7 -> bes

/tmp/ipykernel_39169/4084398841.py:25: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  "skew_nn": float(skew(nn)),


[121/1680] grid_20_0 -> best: GA
[122/1680] grid_20_0 -> best: GA
[123/1680] grid_20_0 -> best: GA
[124/1680] grid_20_0 -> best: LK
[125/1680] grid_20_1 -> best: LK
[126/1680] grid_20_1 -> best: GA
[127/1680] grid_20_1 -> best: LK
[128/1680] grid_20_1 -> best: GA
[129/1680] grid_20_2 -> best: LK
[130/1680] grid_20_2 -> best: GA
[131/1680] grid_20_2 -> best: GA
[132/1680] grid_20_2 -> best: LK
[133/1680] grid_20_3 -> best: GA
[134/1680] grid_20_3 -> best: GA
[135/1680] grid_20_3 -> best: GA
[136/1680] grid_20_3 -> best: GA
[137/1680] grid_20_4 -> best: GA
[138/1680] grid_20_4 -> best: GA
[139/1680] grid_20_4 -> best: GA
[140/1680] grid_20_4 -> best: LK
[141/1680] grid_20_5 -> best: LK
[142/1680] grid_20_5 -> best: GA
[143/1680] grid_20_5 -> best: LK
[144/1680] grid_20_5 -> best: GA
[145/1680] grid_20_6 -> best: LK
[146/1680] grid_20_6 -> best: GA
[147/1680] grid_20_6 -> best: GA
[148/1680] grid_20_6 -> best: GA
[149/1680] grid_20_7 -> best: GA
[150/1680] grid_20_7 -> best: LK
[151/1680]

KeyboardInterrupt: 

In [ ]:
import mealpy
print(mealpy.__version__)


In [ ]:
from mealpy.utils.problem import Problem
import inspect
print(inspect.signature(Problem.__init__))